In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENROUTER_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file"
print("Environment OK.")

Environment OK.


In [11]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    # This is a placeholder tool -- it doesn't call a real weather API,
    # it just returns a fixed string so we can prove the whole pipeline works.
    return f"It's always sunny in {city}!"

agent = create_agent(model="openrouter:nvidia/nemotron-3.5-lightning:free",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result)

{'messages': [HumanMessage(content="What's the weather in San Francisco?", additional_kwargs={}, response_metadata={}, id='30210e99-c31e-4167-9c82-2eb8ffca841a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather for city "San Francisco". Use function get_weather.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to get weather for city "San Francisco". Use function get_weather.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-lightning:free', 'id': 'gen-1788147731-q1bqnpg6qjlUBWGBr4FE', 'created': 1788147731, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a055e9-07d1-77e1-845c-f14c44b03e92-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 

# Initiate a model

In [12]:
from langchain.chat_models import init_chat_model
openrouter_model = init_chat_model('nvidia/nemotron-3.5-lightning:free',
                                   model_provider="openrouter")
response = openrouter_model.invoke("In one sentence, what is LangChain?")
print(response.content)

LangChain is an open-source framework for building context-aware applications by chaining large language model prompts with external tools, data sources, and agents.


In [16]:
print("text:             ", response.text)
print("content_blocks:   ", response.content_blocks)
print("id:               ", response.id)
print("tool_calls:       ", response.tool_calls)
print("content:         ",response.content)
print("metadata:        ",response.response_metadata)

text:              LangChain is an open-source framework for building context-aware applications by chaining large language model prompts with external tools, data sources, and agents.
content_blocks:    [{'type': 'reasoning', 'reasoning': 'Here\'s a thinking process:\n\n1.  **Analyze User Request:**\n   - User asks: "In one sentence, what is LangChain?"\n   - Constraint: Exactly one sentence (or at most one sentence, but typically "in one sentence" means a single concise sentence).\n\n2.  **Identify Key Concepts of LangChain:**\n   - LangChain is a framework/development framework for building applications powered by large language models (LLMs).\n   - It enables chaining/connecting LLMs with other data sources, tools, agents, and prompts.\n   - It supports modular components like prompts, chains, agents, memory, and tools.\n   - Goal: Simplify development of LLM-powered applications.\n\n3.  **Drafting - Attempt 1 (Mental):**\n   LangChain is a framework for developing applications tha

In [15]:
print(response.usage_metadata)

{'input_tokens': 25, 'output_tokens': 495, 'total_tokens': 520, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 511}}


# TripMate Agent


In [12]:
import requests
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
import json
from langchain.chat_models import init_chat_model
from pydantic import BaseModel,Field
import sqlite3
from typing import Literal, Optional,Union
from langchain.agents.structured_output import ToolStrategy,ProviderStrategy
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain.tools import ToolRuntime

In [13]:
load_dotenv()

True

## Weather Tool 

In [14]:
@tool
def get_weather(city: str) -> str:
    """
    Get the current weather information for a city.

    Args:
        city: Name of the city, for example "Vernon", "Toronto", or "Bangalore".

    Returns:
        Current temperature, feels-like temperature, weather condition,
        humidity, wind speed, and city name.
    """

    api_key = os.getenv("OPENWEATHER_API_KEY")

    if not api_key:
        return "Error: OPENWEATHER_API_KEY environment variable is not set."

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }

    try:
        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 404:
            return f"Could not find weather information for '{city}'."

        if response.status_code == 401:
            return "Error: Invalid OpenWeather API key."

        response.raise_for_status()

        data = response.json()

        weather = data["weather"][0]

        result = {
            "city": data["name"],
            "country": data["sys"]["country"],
            "temperature_celsius": data["main"]["temp"],
            "feels_like_celsius": data["main"]["feels_like"],
            "condition": weather["main"],
            "description": weather["description"],
            "humidity_percent": data["main"]["humidity"],
            "wind_speed_mps": data["wind"]["speed"]
        }

        return json.dumps(result)

    except requests.exceptions.Timeout:
        return "Weather API request timed out."

    except requests.exceptions.RequestException as e:
        return f"Weather API request failed: {str(e)}"

    except (KeyError, IndexError):
        return "Unexpected response received from the weather API."

In [67]:
get_weather.invoke({"city":"Vernon"})

'{"city": "Vernon", "country": "CA", "temperature_celsius": 22.36, "feels_like_celsius": 21.83, "condition": "Clouds", "description": "overcast clouds", "humidity_percent": 45, "wind_speed_mps": 1.03}'

### Creating Agent to use the tool

In [14]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", tools=[get_weather])

In [15]:
result = tripmate.invoke({"messages": [{"role": "user", "content": "how are you?"}]})

#### in below response , there won't be any tool call beacuse question is not related to weather

In [16]:
print(result)

{'messages': [HumanMessage(content='how are you?', additional_kwargs={}, response_metadata={}, id='806513d5-dc10-4df5-a4aa-8ed65228af7b'), AIMessage(content="I'm doing great—thank you for asking. How can I help you today?", additional_kwargs={'reasoning_content': 'The user is asking how I am. I should respond politely. No need to use tools.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking how I am. I should respond politely. No need to use tools.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788811071-Hq0koTC4bGlck1DFPefL', 'created': 1788811071, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a07d72-ca5c-7e61-9e7e-70d808a61da5-0', tool_calls=[], invalid_tool_calls=

In [20]:
result = tripmate.invoke({"messages": [{"role": "user", "content": "what is the temperature in Vernon in BC"}]})

In [21]:
print(result)

{'messages': [HumanMessage(content='what is the temperature in Vernon in BC', additional_kwargs={}, response_metadata={}, id='d111dda2-d084-4f1c-954a-51d43b00999b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for the temperature in Vernon, BC. I need to call the get_weather function with city "Vernon". The function requires city name as a string. I\'ll call it.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking for the temperature in Vernon, BC. I need to call the get_weather function with city "Vernon". The function requires city name as a string. I\'ll call it.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788811193-KpOAgdkg15Xfk4Lm7aS6', 'created': 1788811193, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0

## Let's explore weather in Gemma model

In [ ]:
gemma_model = init_chat_model(
    "google/gemma-3-12b-it",
    model_provider="openrouter",
)

In [35]:
@tool
def get_weather_gemma(city: str) -> str:
    """
    Get CURRENT weather information.

    MUST use this tool when the user asks for:
    - current weather
    - temperature
    - humidity
    - wind
    - weather condition
    - whether it is raining
    - weather in a specific city

    Do not answer current weather questions from your own knowledge.

    Args:
        city: The city to get current weather for.
    """
    return f"city {city} has summer"

In [36]:
get_weather_gemma.invoke('Bangalore')

'city Bangalore has summer'

In [43]:
gemma_model_tool = gemma_model.bind_tools([get_weather_gemma])

In [44]:
response = gemma_model_tool.invoke("tell me Vernon BC climate?")
print(response)

content='' additional_kwargs={} response_metadata={'model_name': 'google/gemma-3-12b-it', 'id': 'gen-1788816208-ARsTZs3Dy2j6BzaMleKI', 'created': 1788816208, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.96e-05, 'cost_details': {'upstream_inference_completions_cost': 2.25e-06, 'upstream_inference_prompt_cost': 1.735e-05, 'upstream_inference_cost': 1.96e-05}} id='lc_run--01a07dc1-310b-74f2-b10f-9075e8b89925-0' tool_calls=[{'name': 'get_weather_gemma', 'args': {'city': '{}'}, 'id': 'chatcmpl-tool-94e52eea982c66fd', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 347, 'output_tokens': 15, 'total_tokens': 362, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}


### tool_calls=[{'name': 'get_weather_gemma', 'args': {'city': '{}'}}], gemma unable to extract city name it always brings {}

#### so now add pydantic model with args_schema to get city name

In [52]:
class Weather_city(BaseModel):
    city: str = Field(
        description="The name of the city. Example: Bangalore, Toronto, or Vernon."
    )

In [53]:
@tool(args_schema=Weather_city)
def get_weather_gemma1(city: str) -> str:
    """
    Get CURRENT weather information.

    MUST use this tool when the user asks for:
    - current weather
    - temperature
    - humidity
    - wind
    - weather condition
    - whether it is raining
    - weather in a specific city

    Do not answer current weather questions from your own knowledge.

    Args:
        city: The city to get current weather for.
    """
    return f"city {city} has summer"

In [54]:
get_weather_gemma1.args

{'city': {'description': 'The name of the city. Example: Bangalore, Toronto, or Vernon.',
  'title': 'City',
  'type': 'string'}}

In [68]:
gemma_model_tool = gemma_model.bind_tools([Weather_city])
response = gemma_model_tool.invoke("tell me Vernon city climate?")
print(response)

content='' additional_kwargs={} response_metadata={'model_name': 'google/gemma-3-12b-it', 'id': 'gen-1788820229-gXm9rk13W1nN9lmoy5sT', 'created': 1788820229, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.56e-05, 'cost_details': {'upstream_inference_completions_cost': 1.65e-06, 'upstream_inference_prompt_cost': 1.395e-05, 'upstream_inference_cost': 1.56e-05}} id='lc_run--01a07dfe-8c76-7f00-8683-0c8ed036b521-0' tool_calls=[{'name': 'Weather_city', 'args': {}, 'id': 'chatcmpl-tool-aa36d9d11f69d005', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 279, 'output_tokens': 11, 'total_tokens': 290, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}


In [69]:
gemma_str_op = gemma_model.with_structured_output(get_weather_gemma1)
response = gemma_str_op.invoke("tell me Vernon city climate?")
print(response["city"])

Vernon


### Above city is extracted correctly so we can call tool manually

In [70]:
get_weather.invoke({"city": response["city"]})

'{"city": "Vernon", "country": "CA", "temperature_celsius": 22.36, "feels_like_celsius": 21.83, "condition": "Clouds", "description": "overcast clouds", "humidity_percent": 45, "wind_speed_mps": 1.03}'

In [56]:
gemma_model.profile

{'name': 'Gemma 3 12B',
 'release_date': '2025-03-13',
 'last_updated': '2025-03-13',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'tool_call_streaming': True}

# Pre -Built Tools 

##### This blog defines how to use the pre-defined tools here, for example we are giong to use Tavily search to extract restults from web

In [16]:
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)
@tool('Tavily_Web_Search', description="Use this when the user wants to search the internet.")
def web_search(query:str) -> str:
    """Search the internet for current information."""
    return tavily_search_tool.invoke(query)

In [25]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", tools=[get_weather, web_search])

In [28]:
result = tripmate.invoke({"messages": [{"role": "user", "content": "who is Canada's Prime minister in 2020, use tool to search this"}]})

In [29]:
print(result)

{'messages': [HumanMessage(content="who is Canada's Prime minister in 2020, use tool to search this", additional_kwargs={}, response_metadata={}, id='e64e60f9-0020-4db2-aedb-cde4be51f942'), AIMessage(content='', additional_kwargs={'reasoning_content': "The user is asking about Canada's Prime Minister in 2020. They want me to use a tool to search for this information. I can use the Tavily_Web_Search tool to find the answer. Let me search.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user is asking about Canada's Prime Minister in 2020. They want me to use a tool to search for this information. I can use the Tavily_Web_Search tool to find the answer. Let me search."}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788812725-8IdSA4dvhiPR1Q4YKvse', 'created': 1788812725, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost

# SQLITE3 DB as Tool

In [17]:
DB_NAME = "tripmate.db"

conn = sqlite3.connect(DB_NAME)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS trips (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id TEXT NOT NULL,
    destination TEXT NOT NULL,
    start_date TEXT NOT NULL,
    end_date TEXT NOT NULL,
    status TEXT NOT NULL
)
""")

conn.commit()
conn.close()

print("Database created successfully!")

Database created successfully!


In [18]:
@tool
def save_trip(
    customer_id: str,
    destination: str,
    start_date: str,
    end_date: str,
    status: str = "planned"
) -> str:
    """
    Save a customer's trip in the TripMate database.

    Use this tool when the user wants to save, create,
    or add a trip.

    Args:
        customer_id: Unique customer ID.
        destination: Trip destination.
        start_date: Trip start date in YYYY-MM-DD format.
        end_date: Trip end date in YYYY-MM-DD format.
        status: Trip status, normally 'planned' or 'booked'.
    """

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO trips (
            customer_id,
            destination,
            start_date,
            end_date,
            status
        )
        VALUES (?, ?, ?, ?, ?)
    """, (
        customer_id,
        destination,
        start_date,
        end_date,
        status
    ))

    conn.commit()

    trip_id = cursor.lastrowid

    conn.close()

    return (
        f"Trip saved successfully. "
        f"Trip ID: {trip_id}, "
        f"Customer: {customer_id}, "
        f"Destination: {destination}, "
        f"Dates: {start_date} to {end_date}, "
        f"Status: {status}"
    )


@tool
def get_trip(customer_id: str) -> str:
    """
    Get the trip information for a customer from the TripMate database.

    Use this tool when the user asks about their trip,
    destination, travel dates, or trip status.

    Args:
        customer_id: Unique customer ID.
    """

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            id,
            customer_id,
            destination,
            start_date,
            end_date,
            status
        FROM trips
        WHERE customer_id = ?
        ORDER BY id DESC
        LIMIT 1
    """, (customer_id,))

    trip = cursor.fetchone()

    conn.close()

    if trip is None:
        return f"No trip found for customer {customer_id}."

    return (
        f"Trip ID: {trip[0]}\n"
        f"Customer ID: {trip[1]}\n"
        f"Destination: {trip[2]}\n"
        f"Start Date: {trip[3]}\n"
        f"End Date: {trip[4]}\n"
        f"Status: {trip[5]}"
    )

In [74]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", 
                        tools=[get_weather, web_search , save_trip, get_trip])

In [76]:
result = tripmate.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            My customer ID is CUST001.
            Save my trip to Banff from 2026-09-12 to 2026-09-16.
            The trip is planned.
            """
        }
    ]
})

print(result["messages"][-1].content)

Your trip to **Banff** from **September 12, 2026** to **September 16, 2026** has been saved successfully with a status of **planned** (Trip ID 7). Let me know if you’d like to add any details, change the dates, or book anything!


In [77]:
result = tripmate.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            My customer ID is CUST001.
            What is my trip status?
            """
        }
    ]
})

print(result["messages"][-1].content)

Your trip (Trip ID 7) for customer **CUST001** is currently **planned**. Here are the details:

- **Destination:** Banff  
- **Start Date:** 2026‑09‑12  
- **End Date:** 2026‑09‑16  
- **Status:** **planned**  

Let me know if you’d like to update the status, modify dates, or need any other assistance!


# Adding ToolStrategy into Agents

In [19]:
class NewTripRequest(BaseModel):
    request_type: Literal["trip"]
    customer_id: str = Field(
        description="Unique customer ID"
    )

    destination: str = Field(
        description="Trip destination, for example Banff or Vancouver"
    )

    start_date: str = Field(
        description="Trip start date in YYYY-MM-DD format"
    )

    end_date: str = Field(
        description="Trip end date in YYYY-MM-DD format"
    )

    status: Literal["planned", "booked"] = Field(
        default="planned",
        description="Current status of the trip"
    )


class ModifyTripRequest(BaseModel):
    request_type: Literal["trip"]
    customer_id: str = Field(
        description="Unique customer ID"
    )

    destination: Optional[str] = Field(
        default=None,
        description="New destination if the customer wants to change it"
    )

    start_date: Optional[str] = Field(
        default=None,
        description="New start date in YYYY-MM-DD format if the customer wants to change it"
    )

    end_date: Optional[str] = Field(
        default=None,
        description="New end date in YYYY-MM-DD format if the customer wants to change it"
    )

    status: Optional[Literal["planned", "booked", "cancelled"]] = Field(
        default=None,
        description="New trip status if the customer wants to change it"
    )

class DeleteTripRequest(BaseModel):
    request_type: Literal["trip"]
    customer_id: str = Field(
        description="Unique customer ID"
    )

    reason: Optional[str] = Field(
        default=None,
        description="Optional reason for cancelling the trip"
    )

# class GenericRequest(BaseModel):
#     request_type: Literal["generic"]
#     question: str


In [88]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", 
                        tools=[get_weather, web_search , save_trip, get_trip],
                        response_format=ToolStrategy(Union[DeleteTripRequest,ModifyTripRequest,NewTripRequest]))

In [91]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "can you book a trip to Bali from 1st to 10th october , my ID is 2001"}]
})

{'messages': [HumanMessage(content='can you book a trip to Bali from 1st to 10th october , my ID is 2001', additional_kwargs={}, response_metadata={}, id='256dce3a-7121-4b6f-bb21-d8b4f8add7c8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to book a trip. Use NewTripRequest. Provide destination Bali, start_date 2025-10-01? Wait user didn't specify year. Usually assume current year? Today is 2025-08-14. So upcoming October 2025. Use dates 2025-10-01 to 2025-10-10. status default planned. Use customer_id 2001.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "We need to book a trip. Use NewTripRequest. Provide destination Bali, start_date 2025-10-01? Wait user didn't specify year. Usually assume current year? Today is 2025-08-14. So upcoming October 2025. Use dates 2025-10-01 to 2025-10-10. status default planned. Use customer_id 2001."}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id

In [90]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "what is 4*3"}]
})

{'messages': [HumanMessage(content='what is 4*3', additional_kwargs={}, response_metadata={}, id='650fa91e-7716-4b4e-bb8a-b66918593545'),
  AIMessage(content='[[  {"name": "get_weather", "parameters": {"city": "unknown"}},  {"name": "Tavily_Web_Search", "parameters": {"query": "4*3"}},  {"name": "save_trip", "parameters": {"customer_id": "", "destination": "", "start_date": "", "end_date": "", "status": ""}},  {"name": "get_trip", "parameters": {"customer_id": ""}},  {"name": "DeleteTripRequest", "parameters": {"request_type": "trip", "customer_id": ""}},  {"name": "ModifyTripRequest", "parameters": {"request_type": "trip", "customer_id": ""}},  {"name": "NewTripRequest", "parameters": {"request_type": "trip", "customer_id": "", "destination": "", "start_date": "", "end_date": "", "status": "planned"}}]', additional_kwargs={'reasoning_content': 'The user is asking a simple math question: "what is 4*3". No tool needed. Just answer.', 'reasoning_details': [{'type': 'reasoning.text', 'for

In [92]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "check climate in Bali if it is summerm can you book a trip to Bali from 1st to 10th october , my ID is 2001"}]
})

{'messages': [HumanMessage(content='check climate in Bali if it is summerm can you book a trip to Bali from 1st to 10th october , my ID is 2001', additional_kwargs={}, response_metadata={}, id='5614b08d-b1b8-4918-9e6c-79fefcf77521'),
  AIMessage(content='[[{ "name": "get_weather", "parameters": { "city": "Bali" } } ]', additional_kwargs={'reasoning_content': 'User asks: "check climate in Bali if it is summerm can you book a trip to Bali from 1st to 10th october , my ID is 2001". So they want current climate? Actually "check climate in Bali if it is summerm" probably they want to know if Bali is in summer now. Then book a trip from Oct 1-10, with ID 2001. Use get_weather for Bali to see current climate. Then if they want to book, we need to save_trip with customer_id 2001, destination Bali, start_date 2025-10-01, end_date 2025-10-10, status planned. Need to confirm they want booking. We should first get weather. Then respond with climate info, then ask if they\'d like to book, but they 

In [93]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", 
                        tools=[get_weather, web_search , save_trip, get_trip],
                        # response_format=ToolStrategy(Union[DeleteTripRequest,ModifyTripRequest,NewTripRequest])
                        )

In [94]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "check climate in Bali if it is summerm can you book a trip to Bali from 1st to 10th october , my ID is 2001"}]
})

{'messages': [HumanMessage(content='check climate in Bali if it is summerm can you book a trip to Bali from 1st to 10th october , my ID is 2001', additional_kwargs={}, response_metadata={}, id='7484135d-a972-45bd-8a18-d4e373dd3e1e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to check climate in Bali, specifically if it is summer. Then user wants to book a trip to Bali from 1st to 10th October, with customer ID 2001. We need to check current climate? Or check typical climate? We'll use get_weather for Bali. Then if it is summer (maybe based on season), then book trip using save_trip. We need to check climate first. We'll call get_weather for Bali.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "We need to check climate in Bali, specifically if it is summer. Then user wants to book a trip to Bali from 1st to 10th October, with customer ID 2001. We need to check current climate? Or check typical climate? We'll use

# Forgetting Issue in Agent

In [95]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", 
                        # tools=[get_weather, web_search , save_trip, get_trip],
                        # response_format=ToolStrategy(Union[DeleteTripRequest,ModifyTripRequest,NewTripRequest])
                        )

In [97]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "I am John"}]
})

{'messages': [HumanMessage(content='I am John', additional_kwargs={}, response_metadata={}, id='1cbdcacb-8763-44b1-aef7-778cef5de2d2'),
  AIMessage(content='Hello John! How can I help you today?', additional_kwargs={'reasoning_content': 'The user just introduced themselves as John. This is a simple greeting/introduction. I should respond politely and ask how I can help.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user just introduced themselves as John. This is a simple greeting/introduction. I should respond politely and ask how I can help.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788824350-iOeXvjuTX1x5QyVfHzOG', 'created': 1788824350, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, 

#### Agent can't remeber previous conversation, so we need to implement InMemorySaver which act as checkpoint, it is not just chat history saver, it saves the entire steps, inclusing state, chat etc...

In [98]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "who am I "}]
})

{'messages': [HumanMessage(content='who am I ', additional_kwargs={}, response_metadata={}, id='7c095e9d-cd7d-4f7d-a9ad-093ca4206ae6'),
  AIMessage(content="I don't know who you are.\n\nAs an AI, I don't have access to your personal information, identity, location, or any private data unless you choose to share it with me in our conversation. I only know what you tell me right now.", additional_kwargs={'reasoning_content': 'The user is asking "who am I".\nI am an AI, I do not have access to the user\'s personal identity, location, or private information.\nI need to explain that I don\'t know who they are.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking "who am I".\nI am an AI, I do not have access to the user\'s personal identity, location, or private information.\nI need to explain that I don\'t know who they are.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788824360-3F4reOx

In [20]:
save_memory = InMemorySaver()
thread_config = {"configurable": {"thread_id": "T1"}}


In [112]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", 
                        tools=[get_weather, web_search , save_trip, get_trip],
                        # response_format=ToolStrategy(Union[DeleteTripRequest,ModifyTripRequest,NewTripRequest]),
                        checkpointer=save_memory
                        )



In [113]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "I am John and my ID is J1"}]
}, thread_config)

{'messages': [HumanMessage(content='I am John and my ID is J1', additional_kwargs={}, response_metadata={}, id='4dd43836-dfed-496b-ba6f-b1c499eb55be'),
  AIMessage(content='Hello John! Thanks for letting me know your ID is **J1**. How can I help you today? 😊', additional_kwargs={'reasoning_content': 'User says "I am John and my ID is J1". Probably wants to identify themselves. We may respond acknowledging and ready to help. No tool needed.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'User says "I am John and my ID is J1". Probably wants to identify themselves. We may respond acknowledging and ready to help. No tool needed.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788827518-B0Mi0zTB0XAQzcIAqdEa', 'created': 1788827518, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 

In [114]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "who am I?"}]
},config=thread_config)

{'messages': [HumanMessage(content='I am John and my ID is J1', additional_kwargs={}, response_metadata={}, id='4dd43836-dfed-496b-ba6f-b1c499eb55be'),
  AIMessage(content='Hello John! Thanks for letting me know your ID is **J1**. How can I help you today? 😊', additional_kwargs={'reasoning_content': 'User says "I am John and my ID is J1". Probably wants to identify themselves. We may respond acknowledging and ready to help. No tool needed.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'User says "I am John and my ID is J1". Probably wants to identify themselves. We may respond acknowledging and ready to help. No tool needed.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788827518-B0Mi0zTB0XAQzcIAqdEa', 'created': 1788827518, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 

In [115]:
tripmate.invoke({
    "messages": [{"role": "user", "content": "you know my ID, can you book trip to Banff from Dec 1st to 20 2026? status is confirmed"}]
},config=thread_config)

{'messages': [HumanMessage(content='I am John and my ID is J1', additional_kwargs={}, response_metadata={}, id='4dd43836-dfed-496b-ba6f-b1c499eb55be'),
  AIMessage(content='Hello John! Thanks for letting me know your ID is **J1**. How can I help you today? 😊', additional_kwargs={'reasoning_content': 'User says "I am John and my ID is J1". Probably wants to identify themselves. We may respond acknowledging and ready to help. No tool needed.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'User says "I am John and my ID is J1". Probably wants to identify themselves. We may respond acknowledging and ready to help. No tool needed.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788827518-B0Mi0zTB0XAQzcIAqdEa', 'created': 1788827518, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 

In [ ]:
result = tripmate.invoke({
    "messages": [{"role": "user", "content": "get my banff trip details? and give me ID "}]
},config=thread_config)

In [120]:
result['messages'][-1]

AIMessage(content='Here are your Banff trip details again:\n\n**Trip ID:** **9**  \n**Customer ID:** J1 (John)  \n**Destination:** Banff  \n**Start Date:** 2026‑12‑01  \n**End Date:** 2026‑12‑20  \n**Status:** Confirmed  \n\nLet me know if you’d like to make any changes or need anything else for your trip! 🌲🏔️✨', additional_kwargs={'reasoning_content': 'The user asks again for the banff trip details and ID. We already gave that. We can just respond with the same info.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asks again for the banff trip details and ID. We already gave that. We can just respond with the same info.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788827829-R5TlErVNbLk6w9THtmNb', 'created': 1788827829, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cos

In [21]:
# @title
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage,
    SystemMessage,
)


def pretty_print_chat(state):
    """
    Pretty print a LangChain agent state.

    Args:
        state: Dictionary returned by agent.invoke(...)
    """

    role_icons = {
        HumanMessage: "🧑 Human",
        AIMessage: "🤖 AI",
        ToolMessage: "🛠️ Tool",
        SystemMessage: "⚙️ System",
    }

    print("=" * 100)

    for i, message in enumerate(state["messages"], start=1):

        role = role_icons.get(type(message), type(message).__name__)

        print(f"\n{role} ({i})")
        print("-" * 100)

        if message.content:
            print(message.content)

        # Print tool calls made by the AI
        if isinstance(message, AIMessage) and message.tool_calls:
            print("\n🔧 Tool Calls:")
            for tool in message.tool_calls:
                print(f"   • Tool : {tool['name']}")
                print(f"     Args : {tool['args']}")
                print(f"     ID   : {tool['id']}")

        # Print tool outputs
        if isinstance(message, ToolMessage):
            print(f"\nTool Name    : {message.name}")
            print(f"Tool Call ID : {message.tool_call_id}")

    print("\n" + "=" * 100)

In [122]:
pretty_print_chat(result)


🧑 Human (1)
----------------------------------------------------------------------------------------------------
I am John and my ID is J1

🤖 AI (2)
----------------------------------------------------------------------------------------------------
Hello John! Thanks for letting me know your ID is **J1**. How can I help you today? 😊

🧑 Human (3)
----------------------------------------------------------------------------------------------------
who am I 

🧑 Human (4)
----------------------------------------------------------------------------------------------------
who am I 

🧑 Human (5)
----------------------------------------------------------------------------------------------------
who am I 

🧑 Human (6)
----------------------------------------------------------------------------------------------------
who am I?

🧑 Human (7)
----------------------------------------------------------------------------------------------------
I am John and my ID is J1

🤖 AI (8)
-----------------

# Long-Term Memory

#### agent can store and retrieve information across different conversation, threads and session

In [3]:
long_memory = InMemoryStore()

In [10]:
@tool("save_accommodation_types", description="This function saves user budget preference like luxury, home stay, villa")
def save_budget(user_id:str, trip_type:str, runtime:ToolRuntime) -> str:
    """Save a traveler's preferred trip type (e.g. budget, luxury, adventure) for future visits."""
    runtime.store.put((user_id, "trip"), "budget_type", {"value": trip_type})
    return f"Your budget {trip_type} is saved"

@tool("get_accommodation_types", description="This function get user budget preference like luxury, home stay, villa")
def get_budget(user_id:str, runtime:ToolRuntime) -> str:
    """Recall a traveler's preferred trip style, if saved before."""
    result = runtime.store.get((user_id, "trip"), "budget_type")
    return result.value["value"] if result else f"No preference is saved for user {user_id}"

In [22]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", 
                        tools=[get_weather, web_search , save_trip, get_trip,save_budget,get_budget],
                        # response_format=ToolStrategy(Union[DeleteTripRequest,ModifyTripRequest,NewTripRequest]),
                        checkpointer=save_memory,
                        store=long_memory
                        )

In [26]:
response = tripmate.invoke({
    "messages": [{"role": "user", "content": "I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search"}]
},config=thread_config)

In [27]:
response

{'messages': [HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='59081dd5-a459-4519-9e4b-66a384885934'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='d1349ebe-2213-4c7c-83e5-5b273e690165'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='f9b15597-fe7d-4c88-9d20-30cee3223396'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wa

In [28]:
pretty_print_chat(response)


🧑 Human (1)
----------------------------------------------------------------------------------------------------
I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search

🧑 Human (2)
----------------------------------------------------------------------------------------------------
I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search

🧑 Human (3)
----------------------------------------------------------------------------------------------------
I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search

🤖 AI (4)
------------------------------------------------------------------------------

In [29]:
response = tripmate.invoke({
    "messages": [{"role": "user", "content": "what I requested in Trip earlier?"}]
},config=thread_config)

In [30]:
response

{'messages': [HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='59081dd5-a459-4519-9e4b-66a384885934'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='d1349ebe-2213-4c7c-83e5-5b273e690165'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='f9b15597-fe7d-4c88-9d20-30cee3223396'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wa

In [33]:
response = tripmate.invoke({
    "messages": [{"role": "user", "content": "I requested trip for November but you declined saying August is not fall, can you check again and book for november?"}]
},config=thread_config)

In [34]:
response

{'messages': [HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='59081dd5-a459-4519-9e4b-66a384885934'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='d1349ebe-2213-4c7c-83e5-5b273e690165'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='f9b15597-fe7d-4c88-9d20-30cee3223396'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wa

In [39]:
response = tripmate.invoke({
    "messages": [{"role": "user", "content": "get me trip details in November?"}]
},config=thread_config)

In [40]:
response

{'messages': [HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='59081dd5-a459-4519-9e4b-66a384885934'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='d1349ebe-2213-4c7c-83e5-5b273e690165'),
  HumanMessage(content='I am goudham and ID is G1, can you check what is weather in Kelowna, if it is fall season then book a trip from 1 to 10 november 2026, save my budget is home stay for all future trip search', additional_kwargs={}, response_metadata={}, id='f9b15597-fe7d-4c88-9d20-30cee3223396'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wa

In [41]:
response = tripmate.invoke({
    "messages": [{"role": "user", "content": "what is my saved preference?"}]
},config=thread_config)

In [42]:
response['messages'][-1]

AIMessage(content='Your saved accommodation preference for user **G1** is **“home stay.”** This will be used for any future trip searches unless you change it. Let me know if you’d like to update or add any other preferences!', additional_kwargs={'reasoning_content': 'Now respond with the saved preference.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Now respond with the saved preference.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788838923-s9xfJegf54hiidg5192H', 'created': 1788838923, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a07f1b-cb4d-7673-8ac4-ed2e6fc0c309-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2309, 'output_tokens': 55, 'total_toke